# Recommendation / 01 — Ingestion

Predict simulated customer choice probabilities and rank destination candidates. Use crowd predictions, not actual crowd labels, as features.

Run every cell from top to bottom. Use the project `.venv` kernel in VS Code. No season field is used. Data and test scores are synthetic.

In [1]:
from pathlib import Path
import sys, json, inspect
import pandas as pd
from IPython.display import display
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "ml_service/workflows.py").exists())
sys.path.insert(0, str(ROOT))
from ml_service import workflows as w
print("Project:", ROOT)
print("Synthetic demonstration: real-world accuracy has not been measured.")

Project: /workspace/scratch/71e3a62599e6/YatraSense_Organic
Synthetic demonstration: real-world accuracy has not been measured.


In [2]:
print(inspect.getsource(w.ingest))

def ingest():
    for p in [DATA,MODELS,REPORTS]: p.mkdir(parents=True,exist_ok=True)
    df=pd.read_csv(RAW, keep_default_na=False)
    assert len(df)==14400 and df.destination_id.nunique()==20
    assert df.row_id.is_unique and (df.is_synthetic==1).all()
    assert not any('season' in c for c in df.columns)
    assert set(df.busyness_label)=={'Low','Medium','High'}
    assert set(df.customer_crowd_preference)=={'Low','Medium','High','Any'}
    for a in ACTIVITIES+['customer_'+x for x in ACTIVITIES]: assert df[a].isin([0,1]).all()
    assert (df.visitor_count>=0).all() and df.month.between(1,12).all()
    assert df.temperature_mean_c.between(-50,60).all()
    assert df.precipitation_mm_month.between(0,2000).all()
    pd.to_datetime(df.travel_date, errors='raise')
    # Repeated monthly facts must agree across every customer example.
    facts=['visitor_count','busyness_label','temperature_mean_c','precipitation_mm_month','wind_speed_kmh_mean','popularity_score']
    assert (df.groupby

In [3]:
audit = w.ingest()
display(pd.Series(audit))
df = pd.read_csv(w.DATA / "ingested.csv")
display(df.head())

rows                                                                       14400
destinations                                                                  20
unique_monthly_observations                                                 2400
sha256                         8093405ddbc492618d99dbba9a059ee73b03949b362e71...
synthetic                                                                   True
duplicate_months_expected                                                   True
splits                         {'train': 10080, 'test': 2880, 'validation': 1...
unsupported_activity           wildlife_birdwatching: no tagged destination i...
dtype: object

,row_id,destination_month_id,request_id,destination_id,destination_name,area,destination_type,year,month,travel_date,temperature_mean_c,precipitation_mm_month,wind_speed_kmh_mean,weather_kind,popularity_score,popularity_level,heritage_pilgrimage,trekking_hiking,wildlife_birdwatching,boating_water,adventure_sports,specific_activities,visitor_count,count_scope,busyness_label,customer_heritage_pilgrimage,customer_trekking_hiking,customer_wildlife_birdwatching,customer_boating_water,customer_adventure_sports,customer_crowd_preference,customer_chose_destination,dataset_split,is_synthetic,metadata_status,record_status,source_url,generation_version
0,ROW1,YS001-2016-01,R201601-1,YS001,Pashupatinath,Kathmandu,Temple,2016,1,2016-01-15,7.2,11.2,4.2,synthetic_monthly_conditions,90.3,High,1,0,0,0,0,temple visits; pilgrimage,43412,simulated_visits_not_unique_people,Medium,0,0,1,0,0,Medium,1,train,1,illustrative_activity_tags_not_verified,simulated,NaN,synthetic_demo_v1_seed20260930
1,ROW2,YS001-2016-01,R201601-2,YS001,Pashupatinath,Kathmandu,Temple,2016,1,2016-01-15,7.2,11.2,4.2,synthetic_monthly_conditions,90.3,High,1,0,0,0,0,temple visits; pilgrimage,43412,simulated_visits_not_unique_people,Medium,1,0,0,1,0,High,0,train,1,illustrative_activity_tags_not_verified,simulated,NaN,synthetic_demo_v1_seed20260930
2,ROW3,YS001-2016-01,R201601-3,YS001,Pashupatinath,Kathmandu,Temple,2016,1,2016-01-15,7.2,11.2,4.2,synthetic_monthly_conditions,90.3,High,1,0,0,0,0,temple visits; pilgrimage,43412,simulated_visits_not_unique_people,Medium,0,1,0,1,0,Any,1,train,1,illustrative_activity_tags_not_verified,simulated,NaN,synthetic_demo_v1_seed20260930
3,ROW4,YS001-2016-01,R201601-4,YS001,Pashupatinath,Kathmandu,Temple,2016,1,2016-01-15,7.2,11.2,4.2,synthetic_monthly_conditions,90.3,High,1,0,0,0,0,temple visits; pilgrimage,43412,simulated_visits_not_unique_people,Medium,1,0,0,1,0,Low,0,train,1,illustrative_activity_tags_not_verified,simulated,NaN,synthetic_demo_v1_seed20260930
4,ROW5,YS001-2016-01,R201601-5,YS001,Pashupatinath,Kathmandu,Temple,2016,1,2016-01-15,7.2,11.2,4.2,synthetic_monthly_conditions,90.3,High,1,0,0,0,0,temple visits; pilgrimage,43412,simulated_visits_not_unique_people,Medium,0,1,1,0,1,Medium,0,train,1,illustrative_activity_tags_not_verified,simulated,NaN,synthetic_demo_v1_seed20260930


The input repeats each destination/month for six synthetic customer profiles. This repetition is intentional for recommendation examples; it must not inflate the crowd training dataset.